In [ ]:
# %%
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
AWS_SESSION_TOKEN=

In [ ]:
# secret shhhhh
def get_secret(session):
    """
    Retreive the GitLab secret from AWS SecretsManager

    Returns: Dict, with keys "username" and "password" for GitLab
    """
    secret_name = "uwgitlab/access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Convert raw string into Python dict
    secret = ast.literal_eval(secret)
    return secret

# hf secret shhhhh
def get_secret_hf(session):

    secret_name = "hf-access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return ast.literal_eval(secret)['hf-access-token']

# Create a Secrets Manager session
session = boto3.session.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

git_cred = get_secret(session)
hf_token = get_secret_hf(session)

In [ ]:
# Git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/ka-llm-training.git',
    'username': 'tcwyu',
    'password': git_cred['password']
}

# Script CLI args
hyperparameters = {
    'epochs': 1, 
    # 'lr': 1e-4,
    "seq_len": 8192,
    "weight_decay": 0.1,
    "adam_beta1": 0.9,
    "adam_beta2": 0.95,
    # "learning_rate": 5e-5,
    "learning_rate": 2e-5,
    # "max_grad_norm": 1.0,
    "max_grad_norm": 0.3,
    "gradient_accumulation_steps": 2,
    "gradient_checkpointing": True,
    "deepspeed": 'deepspeed_config.json',
    "output_dir": "/opt/ml/output",
    "per_device_train_batch_size": 2,
    # "per_device_train_batch_size":1,
    "num_train_epochs": 1,
    "fp16": True,
    "fp16_backend": "auto",
    "fp16_full_eval": True,
    "logging_steps": 10,
    "save_total_limit": 1,
    'hf_token': hf_token,
    "training_script": 'LLAMA_KA_Finetuning.py',
    # "training_script": 'LLAMA_KA_Finetuning_LORA_Only.py',
    # "training_script": 'LLAMA_KA_Finetuning_Only.py',
    # "training_script": 'LLAMA_KA_Finetuning_Only_13.py',
    # "training_script": 'Mistral_KA_Finetuning_Only.py',
    # "metric_for_best_model": 'rougeL',
    # "load_best_model_at_end": True,
    "per_device_eval_batch_size": 2,
    # "per_device_eval_batch_size":1,
    "evaluation_strategy": "steps",
    "eval_steps": 19,
    "save_steps": 19,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05
    
}

SAGEMAKER_EXECUTION_ROLE = "arn:aws:iam::506487962374:role/SageMaker-AI-Execution-Role"
ECR_IMAGE_URI = "506487962374.dkr.ecr.ca-central-1.amazonaws.com/ka-llm-training:deepspeed-peft-pretraining"

settings=sagemaker.session_settings.SessionSettings()

sagemaker_session = sagemaker.session.Session(boto_session=session, settings=settings)

estimator_args = {
    # Job config
    "base_job_name": "ka-finetuning",
    "role": SAGEMAKER_EXECUTION_ROLE,
    "output_path": "s3://eko-ekoka-ai-project/output",
    "image_uri": ECR_IMAGE_URI,
    # Instance config
    "instance_count": 1,
    "instance_type": "ml.p3.16xlarge",
    # Script config
    "entry_point": "launch_deepspeed_pt.py",
    "git_config": git_config,
    "source_dir": './scripts',
    "hyperparameters": hyperparameters,
    # Security
    "subnets": ['subnet-0ba981d4c6314f9a9'],
    "security_group_ids": ['sg-04645476ef00795e3'],
    "encrypt_inter_container_traffic": True,
    "sagemaker_session": sagemaker_session,
    "volume_size": 200,
    # "model_uri": "s3://eko-ekoka-ai-project/output/ka-pretraining-2024-03-20-22-06-57-018/output/model.tar.gz"
                # s3://eko-ekoka-ai-project/output/ka-pretraining-2023-12-15-22-05-44-234/output/model.tar.gz
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args, tags=[{"Key":'uw-ai-fine-tuning', "Value":'fine-tuning-mistral'}])

In [ ]:
# Channel - S3 URL pairs for input data to map to within the training instance
input_data = {
    "train": "s3://eko-ekoka-ai-project/data/ft_dataset_generated/train",
    "test": "s3://eko-ekoka-ai-project/data/ft_dataset_generated/test",
    # "train": "s3://eko-ekoka-ai-project/data/ft_dataset_generated_mistral/train",
    # "test": "s3://eko-ekoka-ai-project/data/ft_dataset_generated_mistral/test",
    # "tokenizer": "s3://billsum-prototype/data/tokenizer",
    # "model": "s3://eko-ekoka-ai-project/output/ka-pretraining-2023-12-15-22-05-44-234/output/output.tar.gz"
}

# %%
pytorch_estimator.fit(inputs=input_data)

In [ ]:
# ! aws s3 cp s3://eko-ekoka-ai-project/output/ka-finetuning-2024-03-12-17-18-35-140/output ../../data/lora --sse='AES256' --recursive
# s3://eko-ekoka-ai-project/output/ka-pretraining-2023-12-15-22-05-44-234/output/model.tar.gz

In [ ]:
! aws s3 cp s3://eko-ekoka-ai-project/output/ka-pretraining-2023-12-15-22-05-44-234/output ../../data/pretrained --sse='AES256' --recursive

In [ ]:
pretrained_base_model_uri = "s3://eko-ekoka-ai-project/output/ka-pretraining-2024-03-20-22-06-57-018/output/model.tar.gz"
lora_adaptor_artifact_uri = "s3://eko-ekoka-ai-project/output/ka-finetuning-2024-03-27-17-39-44-884/output/model.tar.gz"

estimator_args_pp = {
    # Job config
    "base_job_name": "ka-merge-models",
    "role": SAGEMAKER_EXECUTION_ROLE,
    "output_path": "s3://eko-ekoka-ai-project/output",
    "image_uri": ECR_IMAGE_URI,
    # Instance config
    "instance_count": 1,
    "instance_type": "ml.p3.16xlarge",
    # Script config
    "entry_point": "post_processing.py",
    # "entry_point": "post_processing_LORA_only.py",
    "git_config": git_config,
    "source_dir": './scripts',
    # Security
    "subnets": ['subnet-0ba981d4c6314f9a9'],
    "security_group_ids": ['sg-04645476ef00795e3'],
    "encrypt_inter_container_traffic": True,
    "sagemaker_session": sagemaker_session,
    "volume_size": 100,
    'hyperparameters':{'pretrained_base_model_uri': pretrained_base_model_uri,
                    'lora_adaptor_artifact_uri': lora_adaptor_artifact_uri,
                    'seq_length':4096,
                    'hf_token':hf_token},
                
}


estimator_pp = sagemaker.estimator.Estimator(**estimator_args_pp, 
                                             tags=[{"Key":'uw-ai-fine-tuning', "Value":'merge'}]
                                             )




In [ ]:

# Start the training job
estimator_pp.fit()



In [ ]:
import tarfile
import numpy as np 

tar = tarfile.open("../../data/merged_model.tar.gz", "r:gz")
for member in tar.getmembers():
     print(member)
        
        

In [ ]:
import tarfile
import os

# /opt/ml/input/data/model/2023_12_15-221854_/output

with tarfile.open("../../data/model.tar.gz") as tar:
    tar.extractall("../../data/merged_model")
    print(f"{os.listdir('../../data/merged_model')}")
    # model = AutoModelForCausalLM.from_pretrained('../../data/model_pretrained')


In [ ]:
from transformers import LlamaTokenizer, LlamaForCausalLM, TrainingArguments, HfArgumentParser, AutoConfig, AutoModelForCausalLM
from peft import PeftModel, PeftConfig, AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained("../../data/lora_model_pretrained")

In [ ]:
directory_path = '../../data/merged_model'
with tarfile.open('../../data/merged_model.tar.gz', "w:gz") as tar:
        tar.add(directory_path, arcname=os.path.basename(directory_path))

In [ ]:
! aws s3 cp ../../data/merged_model.tar.gz s3://eko-ekoka-ai-project/data/merged-model-Mar-19/ --sse='AES256' --recursive



In [ ]:
data/merged_model.tar.gz

In [ ]:
! aws s3 cp s3://eko-ekoka-ai-project/output/ka-finetuning-2024-04-11-17-36-00-889/output/ ../../data/mistral_model.tar.gz --sse='AES256' --recursive



In [ ]:
! tar --append --file=../../data/mistral_model/model.tar.gz ../../data/tokenizer_generation_config.json

In [ ]:
# model = AutoModelForCausalLM.from_pretrained('../../data/model_pretrained/2023_12_15-221854_/output')

In [ ]:
 # ! pip install peft

In [ ]:
import json

# with open('../../data/model_pretrained/2023_12_15-221854_/output/config.json') as file:
#     config = json.load(file)
#     print(config)
    
# with open('../../data/model_pretrained/2023_12_15-221854_/output/config.json', "r", encoding="utf-8") as reader:
#     text = reader.read()
#     config = json.loads(text)
#     print(config)

index_filename = '../../data/model_pretrained/2023_12_15-221854_/output/model.safetensors.index.json'

with open(index_filename, "r") as f:
    index = json.loads(f.read())
    print(index)
